In [13]:

import os
import numpy as np
import torch
import torch.nn as nn

from PIL import Image
from torch.utils.data import Dataset
from torch.utils.data import DataLoader

In [14]:
class ISICDataset(Dataset):

    def __init__(self, image_dir, mask_dir):

        self.image_dir = image_dir
        self.mask_dir = mask_dir

        self.images = sorted(
            [
                f for f in os.listdir(image_dir)
                if f.endswith(".jpg")
            ]
        )

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):

        image_name = self.images[idx]

        image_path = os.path.join(
            self.image_dir,
            image_name
        )

        mask_name = image_name.replace(
            ".jpg",
            "_segmentation.png"
        )

        mask_path = os.path.join(
            self.mask_dir,
            mask_name
        )

        image = Image.open(image_path).resize((256, 256))
        mask = Image.open(mask_path).resize((256, 256))

        image = np.array(image) / 255.0

        mask = np.array(mask)
        mask = (mask > 0).astype(np.uint8)

        image = torch.tensor(
            image,
            dtype=torch.float32
        )

        image = image.permute(2, 0, 1)

        mask = torch.tensor(
            mask,
            dtype=torch.float32
        )

        mask = mask.unsqueeze(0)

        return image, mask

In [15]:
class DoubleConv(nn.Module):

    def __init__(self, in_channels, out_channels):
        super().__init__()

        self.conv = nn.Sequential(

            nn.Conv2d(
                in_channels,
                out_channels,
                kernel_size=3,
                padding=1
            ),

            nn.ReLU(inplace=True),

            nn.Conv2d(
                out_channels,
                out_channels,
                kernel_size=3,
                padding=1
            ),

            nn.ReLU(inplace=True)

        )

    def forward(self, x):
        return self.conv(x)

In [16]:
class EncoderBlock(nn.Module):

    def __init__(self, in_channels, out_channels):
        super().__init__()

        self.conv = DoubleConv(
            in_channels,
            out_channels
        )

        self.pool = nn.MaxPool2d(
            kernel_size=2,
            stride=2
        )

    def forward(self, x):

        features = self.conv(x)

        pooled = self.pool(features)

        return features, pooled

In [17]:
class DecoderBlock(nn.Module):

    def __init__(self, in_channels, out_channels):
        super().__init__()

        self.up = nn.ConvTranspose2d(
            in_channels,
            out_channels,
            kernel_size=2,
            stride=2
        )

        self.conv = DoubleConv(
            out_channels * 2,
            out_channels
        )

    def forward(self, x, skip):

        x = self.up(x)

        x = torch.cat([skip, x], dim=1)

        x = self.conv(x)

        return x

In [18]:
class UNet(nn.Module):

    def __init__(self):
        super().__init__()

        self.enc1 = EncoderBlock(3, 64)
        self.enc2 = EncoderBlock(64, 128)
        self.enc3 = EncoderBlock(128, 256)
        self.enc4 = EncoderBlock(256, 512)

        self.bottleneck = DoubleConv(
            512,
            1024
        )

        self.dec1 = DecoderBlock(
            1024,
            512
        )

        self.dec2 = DecoderBlock(
            512,
            256
        )

        self.dec3 = DecoderBlock(
            256,
            128
        )

        self.dec4 = DecoderBlock(
            128,
            64
        )

        self.final_conv = nn.Conv2d(
            64,
            1,
            kernel_size=1
        )

    def forward(self, x):

        f1, p1 = self.enc1(x)

        f2, p2 = self.enc2(p1)

        f3, p3 = self.enc3(p2)

        f4, p4 = self.enc4(p3)

        b = self.bottleneck(p4)

        d1 = self.dec1(b, f4)

        d2 = self.dec2(d1, f3)

        d3 = self.dec3(d2, f2)

        d4 = self.dec4(d3, f1)

        out = self.final_conv(d4)

        return out

In [19]:
# Create Dataset Object

#Dataset object creates a connection
#between Images folder and Masks folder.
#Total samples = 2594

dataset = ISICDataset(
    r"D:\MedTrack-TX\datasets\ISIC2018_TRAIN\Images",
    r"D:\MedTrack-TX\datasets\ISIC2018_TRAIN\masks"
)

print(len(dataset))

2594


In [20]:
# DataLoader loads data in batches
#Without DataLoader:
#1 image at a time 


train_loader = DataLoader(
    # Dataset source
    dataset,
    # Number of images processed together
    batch_size=16,
    # Shuffle data every epoch
    shuffle=True
)

In [21]:
#lossfunction
# BCEWithLogitsLoss
# BCE = Binary Cross Entropy
# Logits = applies Sigmoid internally

criterion = nn.BCEWithLogitsLoss()

print("Loss Function Ready")

Loss Function Ready


In [22]:
#model
# Create Full U-Net Model
#input img -> u net -> predicted mask
model = UNet()

print("Model Ready")

Model Ready


In [23]:
# Get one batch from DataLoader

images, masks = next(iter(train_loader))

# Pass images through U-Net

outputs = model(images)

# Calculate error between
# predicted mask and actual mask

loss = criterion(
    outputs,
    masks
)

# Check shapes

print("Output Shape :", outputs.shape)
print("Mask Shape   :", masks.shape)

# Print loss value

print("Loss Value   :", loss.item())

Output Shape : torch.Size([16, 1, 256, 256])
Mask Shape   : torch.Size([16, 1, 256, 256])
Loss Value   : 0.7123818397521973


In [24]:
# Adam Optimizer

optimizer = torch.optim.Adam(

    # All model weights
    model.parameters(),

    # Learning Rate
    lr=0.001
)

print("Optimizer Ready")

Optimizer Ready


In [25]:
# Get one batch

images, masks = next(iter(train_loader))

# Clear old gradients

optimizer.zero_grad()

# Forward Pass

outputs = model(images)

# Calculate Loss

loss = criterion(
    outputs,
    masks
)

# Backpropagation

loss.backward()

# Update Weights

optimizer.step()

# Print Loss

print("Training Step Completed")
print("Loss:", loss.item())

Training Step Completed
Loss: 0.7208284139633179


In [ ]:
# Number of times the model sees the full dataset

epochs = 10

for epoch in range(epochs):

    # Store loss of all batches in this epoch

    epoch_loss = 0

    # Loop through all batches

    for images, masks in train_loader:

        # Clear old gradients

        optimizer.zero_grad()

        # Forward Pass

        outputs = model(images)

        # Calculate Loss

        loss = criterion(
            outputs,
            masks
        )

        # Backpropagation

        loss.backward()

        # Update weights

        optimizer.step()

        # Add batch loss

        epoch_loss += loss.item()

    # Average loss of this epoch

    avg_loss = epoch_loss / len(train_loader)

    print(
        f"Epoch [{epoch+1}/{epochs}] "
        f"Loss: {avg_loss:.4f}"
    )

KeyboardInterrupt: 